<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/2_transcribing_audio_and_video_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Transcribing Audio and Video · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. Your numbers will differ a little from run to run, so read the comments as "what
to look for", not as exact results.

Interviews, focus groups, podcasts, parliamentary debates, YouTube videos: a lot of the "text" social
scientists study starts out as **speech**. Transcribing it by hand takes several hours per hour of
audio. Automatic speech recognition has become good enough to do the first pass for you, and the
best-known open model, **Whisper**, runs on your own machine, so recordings never have to leave it.

In this notebook you will:

- generate test recordings with a known transcript, so you can measure accuracy precisely
- transcribe English and Dutch audio with Whisper, with timestamps
- measure the **word error rate** and see why text normalization matters
- compare model sizes (speed vs. accuracy)
- handle long recordings and video files
- learn about Whisper's best-known failure, **hallucination**, and how to catch it
- turn a folder of recordings into a dataset

**Contents**

1. Whisper in a nutshell
2. Making test recordings
3. A first transcription
4. Measuring accuracy: the word error rate
5. Other languages
6. Long recordings, timestamps and video
7. When Whisper makes things up
8. From recordings to a dataset
9. Speakers, privacy and good practice

> ⚠️ Turn on the GPU (*Runtime → Change runtime type → T4 GPU*). Whisper runs on a CPU too, but
> several times slower.

## 0. Setup

We install **openai-whisper** (the official open-source package) and **espeak-ng**, a small
speech synthesizer we use to create test recordings. Colab already has **ffmpeg**, the tool Whisper
uses to read audio and video files.

In [ ]:
!pip install -q openai-whisper
!apt-get install -y -q espeak-ng > /dev/null

import os
import re
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import whisper
from IPython.display import Audio, display
from scipy.io import wavfile

device = "cuda" if torch.cuda.is_available() else "cpu"
AUDIO = Path("audio")
AUDIO.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 120)
print("Whisper models available:", whisper.available_models())
print("Running on", device.upper())

## 1. Whisper in a nutshell

Whisper (Radford et al., 2023) is a transformer, like BERT, but an *encoder-decoder*: the encoder
listens to 30-second windows of audio (turned into a spectrogram, a picture of sound frequencies
over time), and the decoder writes the text, word by word. The original model was trained on 680,000
hours of audio from the web in about 100 languages (later versions on even more), which is why it
copes with accents, background noise and Dutch.

It comes in several sizes. Bigger is more accurate and slower:

| Model | Parameters | Notes |
|---|---|---|
| `tiny` | 39 M | fast, rough; fine for a quick look |
| `base` | 74 M | a reasonable default on a CPU |
| `small` | 244 M | noticeably better; good on a Colab GPU |
| `medium` | 769 M | better still, slower |
| `large-v3` | 1.55 B | the most accurate, needs a good GPU |
| `turbo` | 809 M | close to `large` in quality, much faster |

English-only versions (`base.en`, `small.en`, ...) are slightly better for English, but can't do
anything else.

## 2. Making test recordings

To *measure* accuracy we need recordings whose exact transcript we know. We generate them with
espeak-ng, an open-source speech synthesizer. Its robotic voice is quite unlike a real interview
(no hesitations, no crosstalk, no accent), so accuracy on it says little about your own recordings.
What it gives us is a perfectly known reference, which is what we need to learn the tools.

In [ ]:
def synthesize(text, path, voice="en-us", speed=150):
    # Turn text into a WAV file with espeak-ng (voice: "en-us", "en-gb", "nl", "de", ...)
    subprocess.run(["espeak-ng", "-v", voice, "-s", str(speed), "-w", str(path), text], check=True)
    return path


EN_TEXT = ("I think the new nitrogen rules are necessary, but the government has not explained "
           "them very well. Farmers in my village feel that nobody in The Hague listens to them.")
NL_TEXT = ("De nieuwe stikstofregels zijn nodig, maar de overheid heeft ze niet goed uitgelegd. "
           "Boeren in mijn dorp hebben het gevoel dat niemand in Den Haag naar hen luistert.")

synthesize(EN_TEXT, AUDIO / "interview_en.wav", voice="en-us")
synthesize(NL_TEXT, AUDIO / "interview_nl.wav", voice="nl")

display(Audio(str(AUDIO / "interview_en.wav")))

Whisper reads audio with ffmpeg and converts it to what the model expects: a single channel at
16,000 samples per second. Here's what that looks like:

In [ ]:
samples = whisper.load_audio(str(AUDIO / "interview_en.wav"))     # float array, 16 kHz, mono
print(f"{len(samples):,} samples = {len(samples) / 16000:.1f} seconds")

plt.figure(figsize=(10, 2.2))
plt.plot(np.arange(len(samples)) / 16000, samples, color="#34B233", linewidth=0.4)
plt.xlabel("seconds"); plt.yticks([]); plt.title("The English test recording")
plt.show()

## 3. A first transcription

Loading a model downloads its weights the first time (`base` is about 140 MB). Then one call to
`transcribe` does everything: split the audio into 30-second windows, recognize the speech, and
stitch the pieces together.

In [ ]:
model = whisper.load_model("base", device=device)

start = time.time()
result = model.transcribe(str(AUDIO / "interview_en.wav"), language="en", fp16=(device == "cuda"))
print(f"Transcribed in {time.time() - start:.1f} s\n")
print(result["text"])

Besides the full text, the result contains **segments**: pieces of a few seconds, each with a start
and end time. That's what you need to link a quote back to its place in a recording.

In [ ]:
def segment_table(result, columns=("start", "end", "text", "avg_logprob", "no_speech_prob")):
    # The segments as a table (stays a valid, empty table if Whisper found no speech at all)
    return pd.DataFrame(result["segments"], columns=list(columns)).round(2)


segment_table(result)

Two columns are worth knowing about. `avg_logprob` says how confident the model was in its words
(closer to 0 is more confident), and `no_speech_prob` how likely it thinks the segment contains no
speech at all. Both are useful for flagging segments to check by hand (Section 7).

`fp16=True` uses faster 16-bit arithmetic, which only works on a GPU; on a CPU we switch it off,
which is what `fp16=(device == "cuda")` does.

## 4. Measuring accuracy: the word error rate

The standard measure is the **word error rate** (WER): the minimum number of words to insert,
delete or substitute to turn the transcript into the reference, divided by the number of words in
the reference. It's the word-level version of the character error rate from Bonus 1.

There's a catch: without **normalization**, "Hague," and "hague" count as different words. We
compare both ways.

In [ ]:
def edit_distance(a, b):
    # Levenshtein distance between two sequences (here: lists of words)
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        cur = [i]
        for j, y in enumerate(b, start=1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)          # drop punctuation (keep apostrophes)
    return re.sub(r"\s+", " ", text).strip()


def wer(reference, hypothesis, normalized=True):
    if normalized:
        reference, hypothesis = normalize(reference), normalize(hypothesis)
    ref, hyp = reference.split(), hypothesis.split()
    return edit_distance(ref, hyp) / len(ref)


print(f"WER, raw text:   {wer(EN_TEXT, result['text'], normalized=False):.3f}")
print(f"WER, normalized: {wer(EN_TEXT, result['text']):.3f}")

Our `normalize` only handles case and punctuation. Whisper comes with a more thorough English
normalizer, used in the Whisper paper, that also expands contractions and writes numbers as digits.
It changes what counts as an error:

In [ ]:
from whisper.normalizers import EnglishTextNormalizer

english_normalizer = EnglishTextNormalizer()
ref = "The government has not explained them very well. It affects twenty-three villages."
hyp = "The government hasn't explained them very well. It affects 23 villages."

print("simple normalizer: ", normalize(hyp))
print("Whisper normalizer:", english_normalizer(hyp))
print(f"\nWER with ours:    {wer(ref, hyp):.3f}")
print(f"WER with Whisper's: {wer(english_normalizer(ref), english_normalizer(hyp), normalized=False):.3f}")

So always say how you normalized when you report a WER: the same transcript can score very
differently. (The `jiwer` package computes WER and related measures if you need more than this
function. For Dutch and other languages, `whisper.normalizers.BasicTextNormalizer` does the
language-neutral part.)

> **✏️ Exercise 1**
>
> 1. Synthesize a sentence from your own research area (something with names, numbers or jargon)
>    and transcribe it. Compute the normalized WER.
> 2. Look at the errors. Are they in the words that matter most for your analysis? A transcript with
>    a low WER can still get the key terms wrong.

In [ ]:
# ✅ Solution
my_text = ("Respondents with low trust in science were less likely to get the COVID-19 vaccine, "
           "according to a 2023 survey of 1,500 Dutch adults.")
synthesize(my_text, AUDIO / "my_sentence.wav")
mine = model.transcribe(str(AUDIO / "my_sentence.wav"), language="en", fp16=(device == "cuda"))

print("Reference: ", my_text)
print("Transcript:", mine["text"].strip())
print(f"Normalized WER: {wer(my_text, mine['text']):.3f}")
# Numbers are a classic problem: "1,500" may come back as "1500" or "fifteen hundred", which counts
# as an error even though the meaning is the same. Names and jargon are the other usual suspects.
# Check those by hand, whatever the overall WER says.

> **✏️ Exercise 2**
>
> Transcribe the English recording with **all** of these models: `tiny`, `base` and `small`. For each,
> record the time and the normalized WER, and put the results in a table. Which would you choose for
> 200 hours of interviews?

In [ ]:
# ✅ Solution
rows = []
for size in ["tiny", "base", "small"]:
    m = whisper.load_model(size, device=device)
    start = time.time()
    r = m.transcribe(str(AUDIO / "interview_en.wav"), language="en", fp16=(device == "cuda"))
    rows.append({"model": size, "seconds": time.time() - start, "WER": wer(EN_TEXT, r["text"])})
    del m
display(pd.DataFrame(rows).round(3))
# On one short, clean sentence the differences can be small (and timing includes some start-up
# cost). On long, noisy, accented recordings the bigger models pull clearly ahead. For a large
# project: test two or three sizes on a few minutes of YOUR audio, then choose.

## 5. Other languages

Whisper can **detect the language** itself, or you can tell it with `language=`. Detection looks at
the first 30 seconds only, so with mixed-language recordings it's safer to set the language
explicitly.

In [ ]:
audio_nl = whisper.pad_or_trim(whisper.load_audio(str(AUDIO / "interview_nl.wav")))   # exactly 30 s
mel = whisper.log_mel_spectrogram(audio_nl, n_mels=model.dims.n_mels).to(model.device)
_, probs = model.detect_language(mel)
top3 = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:3]
print("Most likely languages:", [(lang, round(p, 3)) for lang, p in top3])

result_nl = model.transcribe(str(AUDIO / "interview_nl.wav"), language="nl", fp16=(device == "cuda"))
print("\nTranscript:", result_nl["text"].strip())
print(f"Normalized WER: {wer(NL_TEXT, result_nl['text']):.3f}")

Accuracy differs a lot between languages, roughly in line with how much training audio Whisper saw
for each. English is best; Dutch is well supported, but expect more errors, and bigger models help
more for Dutch than for English.

> **✏️ Exercise 3**
>
> Whisper can also **translate**: with `task="translate"` it writes an English translation of
> speech in another language. Transcribe the Dutch recording once with `task="transcribe"` and once
> with `task="translate"`. When could the translation be useful for research, and why should you be
> careful with it?

In [ ]:
# ✅ Solution
for task in ["transcribe", "translate"]:
    r = model.transcribe(str(AUDIO / "interview_nl.wav"), language="nl", task=task, fp16=(device == "cuda"))
    print(f"{task:<11} {r['text'].strip()}")
# Useful for a quick sense of what non-English recordings are about, or to screen a multilingual
# collection. But a translation is an interpretation: nuance, idiom and tone get lost, and you can't
# check the words against the recording. For analysis, transcribe in the original language.

## 6. Long recordings, timestamps and video

Real recordings are long. `transcribe` handles that automatically by moving through the audio in
30-second windows. Let's build a longer "interview" from several answers with pauses in between.

In [ ]:
ANSWERS = [
    "Thank you for having me. I have been a dairy farmer for almost thirty years.",
    "The biggest problem is uncertainty. The rules change every year, so we cannot plan ahead.",
    "I understand that nature needs protection. But the costs should not fall on farmers alone.",
    "My children want to take over the farm, but at the moment I cannot recommend it to them.",
    "What would help is a clear plan for the next twenty years, made together with us.",
]
LONG_TEXT = " ".join(ANSWERS)

pieces = []
for i, answer in enumerate(ANSWERS):
    path = synthesize(answer, AUDIO / f"answer_{i}.wav")
    pieces.append(whisper.load_audio(str(path)))
    pieces.append(np.zeros(int(1.5 * 16000), dtype=np.float32))     # 1.5 seconds of silence
long_audio = np.concatenate(pieces)
wavfile.write(AUDIO / "long_interview.wav", 16000, (long_audio * 32767).astype(np.int16))
print(f"Long interview: {len(long_audio) / 16000:.0f} seconds")

In [ ]:
long_result = model.transcribe(str(AUDIO / "long_interview.wav"), language="en",
                               fp16=(device == "cuda"), word_timestamps=True)
display(segment_table(long_result, ["start", "end", "text"]))
print(f"Normalized WER: {wer(LONG_TEXT, long_result['text']):.3f}")

With `word_timestamps=True`, every word gets its own start and end time. That's handy for linking
transcripts to video, or for measuring things like speaking rate and pauses.

In [ ]:
words = pd.DataFrame([w for seg in long_result["segments"] for w in seg.get("words", [])],
                     columns=["word", "start", "end", "probability"])
words.head(12).round(2)

### Video files

ffmpeg can pull the audio track out of a video, and Whisper can even read most video files
directly. We make a short test video from one of our recordings, then extract its audio again:

In [ ]:
# A 'video': a plain colored screen with our recording as the soundtrack
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "lavfi", "-i", "color=c=0x1A1A2E:s=640x360",
                "-i", str(AUDIO / "interview_en.wav"), "-shortest", "-c:v", "libx264", "-c:a", "aac",
                str(AUDIO / "debate_clip.mp4")], check=True)

# Extract the audio track as 16 kHz mono WAV, the format Whisper uses internally
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(AUDIO / "debate_clip.mp4"),
                "-vn", "-ac", "1", "-ar", "16000", str(AUDIO / "debate_clip.wav")], check=True)

video_result = model.transcribe(str(AUDIO / "debate_clip.mp4"), language="en", fp16=(device == "cuda"))
print("From the video file directly:", video_result["text"].strip())

Extracting the audio first is still useful for long videos: the audio file is much smaller, and you
can keep it as the archived source for your transcript.

## 7. When Whisper makes things up

Whisper's best-known weakness is **hallucination**: during silence, music or noise, it sometimes
writes fluent sentences that nobody said, or repeats the same phrase over and over. Because the
output reads naturally, these errors are easy to miss. Koenecke et al. (2024) found hallucinations
in about 1% of transcriptions of real speech, and some of them were harmful (invented violent or
medical content). Classic speech recognition makes mistakes too, but it doesn't invent whole
sentences.

Let's provoke it: our English recording followed by 20 seconds of quiet background noise.

In [ ]:
rng = np.random.default_rng(0)
speech = whisper.load_audio(str(AUDIO / "interview_en.wav"))
noise = rng.normal(0, 0.01, 20 * 16000).astype(np.float32)          # 20 s of soft hiss
wavfile.write(AUDIO / "with_noise.wav", 16000, (np.concatenate([speech, noise]) * 32767).astype(np.int16))

noisy = model.transcribe(str(AUDIO / "with_noise.wav"), language="en", fp16=(device == "cuda"))
display(segment_table(noisy, ["start", "end", "text", "no_speech_prob"]))

Look at the segments after the speech ends (around the 10-second mark). Anything in them was
invented. Whether you get a hallucination here varies by model and run, but this is the situation
in which they occur. Some defenses:

- **Check segments with a high `no_speech_prob`** or a very low `avg_logprob` by hand.
- Look for **repeated phrases**, a typical hallucination pattern.
- `condition_on_previous_text=False` stops errors from feeding into the next window (at some cost in
  consistency).
- `hallucination_silence_threshold` (together with `word_timestamps=True`) skips long silences.
- Trim long silences and music before transcribing.
- Above all: **spot-check transcripts against the recording**, especially quotes you'll use.

> **✏️ Exercise 4**
>
> 1. Write a function `flag_segments(result)` that returns the segments worth checking by hand: those
>    with `no_speech_prob > 0.5`, or `avg_logprob < -1`, or whose text repeats an earlier segment.
>    Apply it to `noisy`.
> 2. Transcribe `with_noise.wav` again with `condition_on_previous_text=False`,
>    `word_timestamps=True` and `hallucination_silence_threshold=2`. Do the suspicious segments
>    disappear?

In [ ]:
# ✅ Solution
def flag_segments(result, max_no_speech=0.5, min_logprob=-1.0):
    seen, rows = set(), []
    for seg in result["segments"]:
        text = normalize(seg["text"])
        reasons = []
        if seg["no_speech_prob"] > max_no_speech:
            reasons.append("probably no speech")
        if seg["avg_logprob"] < min_logprob:
            reasons.append("low confidence")
        if text and text in seen:
            reasons.append("repeats earlier text")
        seen.add(text)
        if reasons:
            rows.append({"start": seg["start"], "end": seg["end"], "text": seg["text"],
                         "why": ", ".join(reasons)})
    return pd.DataFrame(rows, columns=["start", "end", "text", "why"])

display(flag_segments(noisy))

careful = model.transcribe(str(AUDIO / "with_noise.wav"), language="en", fp16=(device == "cuda"),
                           condition_on_previous_text=False, word_timestamps=True,
                           hallucination_silence_threshold=2)
display(segment_table(careful, ["start", "end", "text", "no_speech_prob"]))
# The flags are a triage tool: they point you to the parts worth listening to. The safer settings
# reduce hallucinations, but nothing replaces checking the recording for the passages you quote.

## 8. From recordings to a dataset

As with the PDF pipeline in Bonus 1: walk through a folder, transcribe each file, and collect the
segments in one table, with the file name and timestamps, ready for coding or for any method from
the course.

In [ ]:
def transcribe_folder(folder, language=None, pattern="*.wav"):
    rows, problems = [], []
    for path in sorted(Path(folder).glob(pattern)):
        try:
            r = model.transcribe(str(path), language=language, fp16=(device == "cuda"))
            for seg in r["segments"]:
                rows.append({"file": path.name, "language": r["language"], "start": round(seg["start"], 1),
                             "end": round(seg["end"], 1), "text": seg["text"].strip(),
                             "no_speech_prob": round(seg["no_speech_prob"], 3)})
        except Exception as e:                                # one bad file shouldn't stop the run
            problems.append({"file": path.name, "error": f"{type(e).__name__}: {e}"})
    return pd.DataFrame(rows), pd.DataFrame(problems)


corpus, problems = transcribe_folder(AUDIO, pattern="answer_*.wav", language="en")
corpus

> **✏️ Exercise 5**
>
> Extend `transcribe_folder` so that it also records, for each file, the **duration** of the
> recording and the **time** transcription took. Then compute the *real-time factor* (transcription
> time divided by audio duration) for the folder. How long would 100 hours of interviews take at
> that speed?

In [ ]:
# ✅ Solution
def transcribe_folder_timed(folder, language=None, pattern="*.wav"):
    rows, files = [], []
    for path in sorted(Path(folder).glob(pattern)):
        duration = len(whisper.load_audio(str(path))) / 16000
        start = time.time()
        r = model.transcribe(str(path), language=language, fp16=(device == "cuda"))
        took = time.time() - start
        files.append({"file": path.name, "duration_s": duration, "transcription_s": took})
        rows += [{"file": path.name, "start": s["start"], "end": s["end"], "text": s["text"].strip()}
                 for s in r["segments"]]
    return pd.DataFrame(rows), pd.DataFrame(files)

segs, timing = transcribe_folder_timed(AUDIO, pattern="answer_*.wav", language="en")
display(timing.round(2))
rtf = timing["transcription_s"].sum() / timing["duration_s"].sum()
print(f"Real-time factor: {rtf:.2f}  ->  100 hours of audio would take about {100 * rtf:.0f} hours")
# Very short files overstate the cost (fixed overhead per file). Measure on a few full-length
# recordings before planning a big job; on a GPU, Whisper is usually much faster than real time.

## 9. Speakers, privacy and good practice

### Who said what?

Whisper doesn't know *who* is speaking. Splitting a recording by speaker is called **speaker
diarization**. The most common open tool is `pyannote.audio`; **WhisperX** combines Whisper with
pyannote and adds precise word timing. Pyannote's models require a (free) Hugging Face account and
accepting their terms. For two-person interviews, many researchers still assign speakers by hand,
which is quick once the text is there.

### Faster options

- **faster-whisper**: the same models, reimplemented to run several times faster with less memory.
- The Hugging Face `transformers` pipeline (`pipeline("automatic-speech-recognition",
  model="openai/whisper-small")`), if you prefer the tools from Day 3.

### Privacy and ethics

- **Voices are personal data**, and interview content is often sensitive. Get consent for
  recording *and* for automatic transcription, and follow your data management plan.
- Whisper runs **locally**, which is a big advantage over online transcription services. Colab,
  however, runs on Google's servers: for confidential recordings, run Whisper on your own computer
  or your university's infrastructure.
- Anonymize transcripts before sharing them (Bonus 4).

### A workflow that holds up

1. Test two or three model sizes on a few minutes of your own audio; measure the WER against a
   hand-made transcript.
2. Transcribe everything with the chosen model, keeping timestamps.
3. Flag suspicious segments and check them, plus a random sample, against the recording.
4. Correct the transcripts you'll quote from; report the model, settings and error rate.

### Further reading

- Radford, A., Kim, J. W., Xu, T., Brockman, G., McLeavey, C., & Sutskever, I. (2023). Robust speech
  recognition via large-scale weak supervision. *Proceedings of ICML 2023*. (Whisper.)
- Koenecke, A., Choi, A. S. G., Mei, K. X., Schellmann, H., & Sloane, M. (2024). Careless Whisper:
  Speech-to-text hallucination harms. *Proceedings of ACM FAccT 2024*.
- Bain, M., Huh, J., Han, T., & Zisserman, A. (2023). WhisperX: Time-accurate speech transcription of
  long-form audio. *Proceedings of Interspeech 2023*.